# Ejercicio 4 — Diseño $3^3$ completo y análisis RSM (R)

**Objetivo.** Analizar un $3^3$ (27 corridas) con ANOVA por componentes L/Q,
ajustar modelo de segundo orden y localizar el óptimo.

**Factores:** Agua, Cemento, Arena (en kg/m³)
**Respuesta:** Resistencia a compresión (MPa)

In [ ]:
library(dplyr)
library(ggplot2)
library(rsm)

df <- read.csv('../../datos/resistencia-concreto-3k3.csv')
cat(sprintf('Corridas: %d\n', nrow(df)))
print(head(df))

## 1. ANOVA

In [ ]:
modelo <- lm(resistencia ~ x1+x2+x3+I(x1^2)+I(x2^2)+I(x3^2)+x1:x2+x1:x3+x2:x3, data=df)
print(anova(modelo))

## 2. Contrastes L/Q

In [ ]:
contrastes_lq <- function(df, factor, resp='resistencia') {
  med <- tapply(df[[resp]], df[[factor]], mean)
  n   <- table(df[[factor]])[1]
  CL  <- med['1'] - med['-1']
  CQ  <- med['-1'] - 2*med['0'] + med['1']
  # Con C calculado de medias: SC = n_nivel * C_medias² / Σc²  (Σc²=2 para L, 6 para Q)
  c(C_L=round(CL,3), C_Q=round(CQ,3),
    SC_L=round(n*CL^2/2,3), SC_Q=round(n*CQ^2/6,3))
}
tab <- t(sapply(c('x1','x2','x3'), contrastes_lq, df=df))
rownames(tab) <- c('A(agua)','B(cemento)','C(arena)')
print(tab)
cat('\nVerificación: compare SC_L + SC_Q de cada factor con su SS en anova(modelo).\n')

## 3. Análisis canónico

In [ ]:
modelo_rsm <- rsm(resistencia ~ SO(x1,x2,x3), data=df)
cat('Punto estacionario:\n')
can <- canonical(modelo_rsm)
print(can)

xs <- can$xs
x_real <- c(170,350,700) + xs * c(30,50,100)
cat(sprintf('\nÓptimo real: agua=%.0f, cemento=%.0f, arena=%.0f\n', x_real[1],x_real[2],x_real[3]))
eigs <- can$eigen$values
cat(sprintf('Tipo de punto: %s\n', if(all(eigs<0)) 'MÁXIMO' else if(all(eigs>0)) 'MÍNIMO' else 'SILLA'))

## 4. Curvas de nivel

In [ ]:
options(repr.plot.width=13, repr.plot.height=5)
par(mfrow=c(1,3))
contour(modelo_rsm, x1~x2, at=list(x3=xs['x3']), image=TRUE,
        col.image=terrain.colors(20), main='x3 en óptimo', xlab='x1',ylab='x2')
contour(modelo_rsm, x1~x3, at=list(x2=xs['x2']), image=TRUE,
        col.image=terrain.colors(20), main='x2 en óptimo', xlab='x1',ylab='x3')
contour(modelo_rsm, x2~x3, at=list(x1=xs['x1']), image=TRUE,
        col.image=terrain.colors(20), main='x1 en óptimo', xlab='x2',ylab='x3')

## 5. Conclusión

- `rsm::SO()` ajusta el modelo de segundo orden completo con todos los términos L, Q y bilineales.
- `canonical()` proporciona el punto estacionario y los eigenvalores que clasifican el óptimo.
- Para 3+ factores, los contornos por pares (fijando el tercero) son la forma más clara de
  visualizar la superficie de respuesta.